In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & NHÃN
# ==========================================
BASE_DIR = Path(__file__).resolve().parent
TEST_CSV = BASE_DIR / "dataset" / "test.csv"  # Sửa đường dẫn nếu file test nằm chỗ khác
MODEL_PATH = BASE_DIR / "models" / "visobert_oversampled_best"
FALLBACK_TOKENIZER = "5CD-AI/Vietnamese-Sentiment-visobert"

OUTPUT_DIR = BASE_DIR / "evaluation_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

TEXT_COL = "comment"
LABEL_COL = "label"
MAX_LENGTH = 128
BATCH_SIZE = 16

# Ánh xạ nhãn đồng bộ với toàn đồ án (0: Tiêu cực, 1: Trung tính, 2: Tích cực)
ID2LABEL = {0: "Tiêu cực", 1: "Trung tính", 2: "Tích cực"}
LABEL_IDS = sorted(ID2LABEL.keys())
LABEL_NAMES = [ID2LABEL[i] for i in LABEL_IDS]


def load_model_and_tokenizer(model_path: Path, device: torch.device):
  try:
    tokenizer = AutoTokenizer.from_pretrained(str(model_path), use_fast=False)
  except Exception:
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_TOKENIZER, use_fast=False)

  model = AutoModelForSequenceClassification.from_pretrained(str(model_path))
  model.eval()
  model.to(device)
  return tokenizer, model


def run_test_inference(texts: list[str], tokenizer, model, device: torch.device):
  if device.type == "cpu":
    torch.set_num_threads(max(1, torch.get_num_threads()))

  all_probs = []
  total_batches = (len(texts) + BATCH_SIZE - 1) // BATCH_SIZE

  with torch.inference_mode():
    for batch_idx, start in enumerate(range(0, len(texts), BATCH_SIZE), start=1):
      batch_texts = texts[start : start + BATCH_SIZE]
      encoded = tokenizer(
          batch_texts,
          return_tensors="pt",
          truncation=True,
          padding=True,
          max_length=MAX_LENGTH,
      ).to(device)

      outputs = model(**encoded)
      probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
      all_probs.append(probs)

      if batch_idx % 20 == 0 or batch_idx == total_batches:
        print(f"Đã xử lý {batch_idx}/{total_batches} batch...")

  probs_arr = np.vstack(all_probs)
  preds_arr = np.argmax(probs_arr, axis=-1)
  return preds_arr, probs_arr


def save_matrix_image(cm: np.ndarray, save_path: Path):
  plt.figure(figsize=(7, 6))
  sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      cmap="Blues",
      xticklabels=LABEL_NAMES,
      yticklabels=LABEL_NAMES,
  )
  plt.title("Confusion Matrix - ViSoBERT (Test Set)", fontsize=13, pad=12)
  plt.xlabel("Nhãn dự đoán (Predicted)", fontsize=11)
  plt.ylabel("Nhãn thực tế (True)", fontsize=11)
  plt.tight_layout()
  plt.savefig(save_path, dpi=300)
  plt.close()


def main():
  if not TEST_CSV.exists():
    raise FileNotFoundError(f"Không tìm thấy file test: {TEST_CSV}")

  df = pd.read_csv(TEST_CSV)
  texts = df[TEXT_COL].astype(str).tolist()
  y_true = df[LABEL_COL].astype(int).to_numpy()

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Đang chạy trên thiết bị: {device} | Số mẫu test: {len(texts)}")

  tokenizer, model = load_model_and_tokenizer(MODEL_PATH, device)
  y_pred, probs_arr = run_test_inference(texts, tokenizer, model, device)

  # 1. Tính và in Confusion Matrix
  cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
  cm_df = pd.DataFrame(
      cm,
      index=[f"True: {name}" for name in LABEL_NAMES],
      columns=[f"Pred: {name}" for name in LABEL_NAMES],
  )

  print("\n=== CONFUSION MATRIX (TẬP TEST) ===")
  print(cm_df)

  print("\n=== CLASSIFICATION REPORT ===")
  print(
      classification_report(
          y_true, y_pred, labels=LABEL_IDS, target_names=LABEL_NAMES, digits=4
      )
  )

  # 2. Lưu ảnh ma trận và các dòng sai
  img_path = OUTPUT_DIR / "visobert_confusion_matrix.png"
  save_matrix_image(cm, img_path)

  df["pred_label"] = y_pred
  df["true_label_name"] = df[LABEL_COL].map(ID2LABEL)
  df["pred_label_name"] = df["pred_label"].map(ID2LABEL)
  for i in range(probs_arr.shape[1]):
    df[f"prob_{i}"] = np.round(probs_arr[:, i], 4)

  errors_df = df[df[LABEL_COL] != df["pred_label"]]
  errors_path = OUTPUT_DIR / "error_analysis_visobert.csv"
  errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

  print(f"\nĐã lưu ma trận ảnh tại : {img_path}")
  print(
      f"Đã lưu {len(errors_df)}/{len(df)} dòng lỗi tại : {errors_path}"
  )


if __name__ == "__main__":
  main()
